In [1]:
#imports
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split

In [2]:
#load dataset
def load_dataset(path):
    df = pd.read_csv(path)
    print("Dataset loaded:", df.shape)
    return df

In [3]:
#load rules (rules.json must be valid JSON)
def load_rules(path):
    with open(path, "r") as f:
        rules = json.load(f)
    print("Rules loaded")
    return rules

In [4]:
#ensure hidden column exists (in both train and test)
def ensure_hidden_column(df, rules):
    # assumes rules["variables"] exists and lists hidden variables
    hidden_vars = list(rules.get("variables", {}).keys())
    for hv in hidden_vars:
        if hv not in df.columns:
            df[hv] = np.nan
    return df

In [5]:
#apply rules to a dataframe (do NOT apply rules to test set)
def apply_rules(df, rules):
    for rule in rules.get("rules", []):
        condition = rule["condition"]
        mask = df.eval(condition)
        for var, val in rule["assign"].items():
            # FIX: only assign where var is still NaN (first-match-only)
            df.loc[mask & df[var].isna(), var] = val
    return df


In [6]:
# =========================
# CONFIG (GLOBAL SETTINGS)
# =========================

CONFIG = {
    # Paths
    "dataset_path": r"D:\project MODEL.T\dataset\Master_data.csv",
    "rules_path": r"D:\project MODEL.T\rule.json\NEW_RULE.json",
    "output_path": r"D:\project MODEL.T\dataset\model_T_results6.csv",

    # Split
    "test_size": 0.2,

    # Reproducibility
    "random_seed": 42,

    # Training params  
    "learning_rate": 0.001,
    "epochs": 300,
    "batch_size": 32,

    # Model / similarity params (keep for future use)
    "k_neighbors": 10,
    "buffer_pct": 0.10,
    "similarity_sigma": 1.0,
}

In [7]:
# load data and split
df = load_dataset(CONFIG["dataset_path"])
rules = load_rules(CONFIG["rules_path"])

# create hidden column placeholders in full df so test set has column too
df = ensure_hidden_column(df, rules)

# train/test split - test set must be held-out
train_df, test_df = train_test_split(
    df,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_seed"]
)

print("Train:", len(train_df), "Test:", len(test_df))

Dataset loaded: (1000, 6)
Rules loaded
Train: 800 Test: 200


In [8]:
#quality check

import numpy as np
from numpy.linalg import matrix_rank, cond as matrix_cond

TARGET_COL = "target"

# ── Step 1: apply rules to a temp copy (never touch real train_df) ──
df_temp = train_df.copy()
hv      = list(rules["variables"].keys())[0]
h_range = rules["variables"][hv]["range"]

if hv not in df_temp.columns:
    df_temp[hv] = np.nan

for rule in rules["rules"]:
    mask = df_temp.eval(rule["condition"])
    for var, val in rule["assign"].items():
        df_temp.loc[mask & df_temp[var].isna(), var] = val

labeled   = df_temp[df_temp[hv].notna()].copy()
unlabeled = df_temp[df_temp[hv].isna()].copy()

# ── Step 2: build feature matrix from labeled rows ──
features_chk = [c for c in df_temp.select_dtypes(include=[np.number]).columns
                if c != TARGET_COL]

N     = len(df_temp)
n_lab = len(labeled)
n_unl = len(unlabeled)
pct   = n_lab / N * 100

X_lab = labeled[features_chk].values.astype(float)

# ── Step 3: compute rank ──
rank   = matrix_rank(X_lab)
nfeats = len(features_chk)

# ── Step 4: compute condition number (on normalized matrix) ──
fmean = X_lab.mean(axis=0)
fstd  = X_lab.std(axis=0)
fstd[fstd == 0] = 1e-9
X_norm_chk = (X_lab - fmean) / fstd
try:
    cond_num = matrix_cond(X_norm_chk)
except Exception:
    cond_num = float("inf")

# ── Step 5: per-feature std in labeled set ──
feat_stds  = {f: X_lab[:, i].std() for i, f in enumerate(features_chk)}
zero_feats = [f for f, s in feat_stds.items() if s < 0.01]

# ── Step 6: hidden variable value distribution ──
hv_dist   = labeled[hv].value_counts().sort_index().to_dict()
hv_unique = len(hv_dist)

# ── Step 7: evaluate each check ──
chk_coverage = pct >= 15
chk_rank     = rank == nfeats
chk_cond     = cond_num < 100
chk_zero     = len(zero_feats) == 0
chk_hv       = hv_unique >= 2
all_pass     = chk_coverage and chk_rank and chk_cond and chk_zero and chk_hv

def tick(b): return "PASS" if b else "FAIL"
def cq(v):   return "excellent" if v<10 else "good" if v<30 else "poor" if v<100 else "CRITICAL"

# ── Print report ──
sep = "-" * 55

print(sep)
print("  RULE QUALITY CHECK")
print(sep)

print(f"\n  Rows total      : {N}")
print(f"  Labeled rows    : {n_lab}  ({pct:.1f}%)")
print(f"  Unlabeled rows  : {n_unl}")
print(f"  Features        : {features_chk}")

print(f"\n  {sep}")

# Check 1 — coverage
print(f"  CHECK 1  Coverage >= 15%")
print(f"  Result : {tick(chk_coverage)}  ({pct:.1f}% labeled)")
if not chk_coverage:
    print(f"  Fix    : add more rules to label at least 15% of rows")
print()

# Check 2 — rank  (most important)
print(f"  CHECK 2  Matrix rank == number of features")
print(f"  Result : {tick(chk_rank)}  rank = {rank}  /  n_features = {nfeats}")
if not chk_rank:
    print(f"  Fix    : your rules fix a linear constraint across features")
    print(f"           (e.g. m1+m2+m3+m4 always = 3 in labeled rows)")
    print(f"           add rules with different sums or laser thresholds")
    print(f"           so labeled rows have real variance in all features")
print()

# Check 3 — condition number
cond_str = f"{cond_num:.1f}" if cond_num < 1e10 else f"{cond_num:.2e}"
print(f"  CHECK 3  Condition number < 100")
print(f"  Result : {tick(chk_cond)}  {cond_str}  [{cq(cond_num)}]")
if not chk_cond:
    print(f"  Fix    : same as Check 2 — add diverse rules")
print()

# Check 4 — per-feature variance
print(f"  CHECK 4  Every feature has variance > 0 in labeled rows")
print(f"  Result : {tick(chk_zero)}")
for f, s in feat_stds.items():
    bar  = chr(9608) * min(int(s * 15), 30)
    flag = "  <- near zero" if s < 0.01 else ""
    print(f"           {f:<10}  std = {s:.4f}   {bar}{flag}")
if not chk_zero:
    print(f"  Fix    : features {zero_feats} never change in labeled rows")
    print(f"           add rules where those features take different values")
print()

# Check 5 — hidden variable variety
print(f"  CHECK 5  Hidden variable has >= 2 distinct values in labeled rows")
print(f"  Result : {tick(chk_hv)}  ({hv_unique} distinct values)")
for val, cnt in hv_dist.items():
    print(f"           {hv} = {val}  ->  {cnt} rows")
if not chk_hv:
    print(f"  Fix    : all labeled rows assign the same {hv} value")
    print(f"           add at least one rule that assigns a different value")

print(f"\n  {sep}")
if all_pass:
    print(f"  ALL CHECKS PASSED")
    print(f"  rules are usable — proceed with training")
else:
    print(f"  ONE OR MORE CHECKS FAILED")
    print(f"  fix the rules in your JSON file and re-run Cell A then this cell")
print(sep)

-------------------------------------------------------
  RULE QUALITY CHECK
-------------------------------------------------------

  Rows total      : 800
  Labeled rows    : 234  (29.2%)
  Unlabeled rows  : 566
  Features        : ['m1', 'm2', 'm3', 'm4', 'laser', 'm5']

  -------------------------------------------------------
  CHECK 1  Coverage >= 15%
  Result : PASS  (29.2% labeled)

  CHECK 2  Matrix rank == number of features
  Result : PASS  rank = 6  /  n_features = 6

  CHECK 3  Condition number < 100
  Result : PASS  5.6  [excellent]

  CHECK 4  Every feature has variance > 0 in labeled rows
  Result : PASS
           m1          std = 0.4988   ███████
           m2          std = 0.4998   ███████
           m3          std = 0.4999   ███████
           m4          std = 0.4998   ███████
           laser       std = 6.0359   ██████████████████████████████
           m5          std = 0.9655   ██████████████

  CHECK 5  Hidden variable has >= 2 distinct values in labeled r

In [9]:
#apply rules ONLY on training set, ensure test still has hidden column
train_df = apply_rules(train_df, rules)
test_df = ensure_hidden_column(test_df, rules)   # keep structure consistent

In [10]:
#identify hidden variable and split labeled/unlabeled (train only)
hidden_var = list(rules["variables"].keys())[0]   # general: first hidden variable
labeled_rows = train_df[train_df[hidden_var].notna()]
unlabeled_rows = train_df[train_df[hidden_var].isna()]

print("Train total:", len(train_df))
print("Labeled (rule rows):", len(labeled_rows))
print("Unlabeled:", len(unlabeled_rows))

# FIX: Save a clean snapshot of train_df BEFORE h-recovery
# Baselines must impute h themselves from this clean copy
train_df_clean = train_df.copy()


Train total: 800
Labeled (rule rows): 234
Unlabeled: 566


In [11]:


# select numeric features automatically (general)

target = "target"

numeric_cols = list(train_df.select_dtypes(include=[np.number]).columns)

# ✅ KEEP hidden_var (this is the key fix)
features = [c for c in numeric_cols if c not in [target, "is_rule"]]

print("Features used for base model:", features)

Features used for base model: ['m1', 'm2', 'm3', 'm4', 'laser', 'm5']


In [12]:
# initialize weights, bias, hyperparams

n_features = len(features)

# small random initialization
weights = np.random.randn(n_features) * 0.01
bias = 0.0

# hyperparameters from CONFIG
learning_rate = CONFIG["learning_rate"]
epochs = CONFIG["epochs"]
batch_size = CONFIG["batch_size"]

In [13]:
#loss calculator
def predict(X, weights, bias):  
    
    return np.dot(X, weights) + bias


def mse_loss(y_true, y_pred):
    """
    Compute Mean Squared Error loss."""
    return np.mean((y_true - y_pred)**2)

In [14]:
# train ONLY on rows where hidden_var is known
train_labeled = train_df[train_df[hidden_var].notna()]

X_train = train_labeled[features].values.astype(float)
y_train = train_labeled[target].values.astype(float)

# ---------- NORMALIZATION (MANDATORY) ----------
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_std[X_std == 0] = 1.0

X_train = (X_train - X_mean) / X_std

y_mean = y_train.mean()
y_std = y_train.std()
y_train = (y_train - y_mean) / y_std
# ----------------------------------------------

batch_size = min(batch_size, len(X_train))
loss_history = [] 

for epoch in range(epochs):

    perm = np.random.permutation(len(X_train))
    X_sh = X_train[perm]
    y_sh = y_train[perm]

    for i in range(0, len(X_sh), batch_size):

        X_batch = X_sh[i:i+batch_size]
        y_batch = y_sh[i:i+batch_size]

        y_pred = predict(X_batch, weights, bias)
        error = y_pred - y_batch

        dw = (2.0 / len(y_batch)) * np.dot(X_batch.T, error)
        db = (2.0 / len(y_batch)) * np.sum(error)

        # ---------- GRADIENT CLIPPING ----------
        dw = np.clip(dw, -1, 1)
        db = np.clip(db, -1, 1)
        # --------------------------------------

        weights -= learning_rate * dw
        bias -= learning_rate * db

    if epoch % 50 == 0:
        loss = mse_loss(y_train, predict(X_train, weights, bias))
        loss_history.append(loss)   
        print(f"Epoch {epoch:04d} loss: {loss:.6f}")

Epoch 0000 loss: 0.999693
Epoch 0050 loss: 0.904111
Epoch 0100 loss: 0.878668


Epoch 0150 loss: 0.860837
Epoch 0200 loss: 0.845513
Epoch 0250 loss: 0.832249


In [15]:
# estimate hidden on unlabeled train rows (your method)

unlabeled_rows = train_df[train_df[hidden_var].isna()].copy()

if len(unlabeled_rows) > 0:

    feature_index = {f: i for i, f in enumerate(features)}

    # 🔒 safety check
    if hidden_var not in feature_index:
        raise ValueError(
            f"{hidden_var} not found in features.\n"
            f"Current features: {features}\n"
            f"👉 You must include hidden_var in features before training."
        )

    hidden_idx = feature_index[hidden_var]
    w_hidden = weights[hidden_idx]

    # Step 1 — base prediction (set hidden = 0)
    X_unlabeled = unlabeled_rows[features].copy()
    X_unlabeled[hidden_var] = 0

    X_raw_unl = X_unlabeled.values.astype(float)

    # safe normalization
    X_std_safe = np.where(X_std == 0, 1.0, X_std)
    X_norm_unl = (X_raw_unl - X_mean) / X_std_safe

    base_pred = predict(X_norm_unl, weights, bias)

    # convert to real scale
    base_pred_real = base_pred * y_std + y_mean

    # Step 2 — convert hidden weight to real scale
    if X_std_safe[hidden_idx] == 0:
        print("⚠️ hidden feature std is zero → unstable scaling")
        w_hidden_real = 1e-8
    else:
        w_hidden_real = w_hidden * (y_std / X_std_safe[hidden_idx])

    # safety check for near-zero weight
    epsilon = 1e-8
    if abs(w_hidden_real) < epsilon:
        print("⚠️ Hidden weight too small — unstable recovery")
        w_hidden_real = epsilon

    # Step 3 — solve for hidden
    hidden_est = (unlabeled_rows[target].values - base_pred_real) / w_hidden_real

    # Step 4 — clip to valid range
    low, high = rules["variables"][hidden_var]["range"]
    hidden_est = np.clip(hidden_est, low, high)

    # Step 5 — write back
    train_df.loc[unlabeled_rows.index, hidden_var] = hidden_est

    print(f"Filled {len(hidden_est)} unlabeled train rows.")
    print(f"Recovered range: [{hidden_est.min():.4f}, {hidden_est.max():.4f}]")
    print(f"Expected range: [{low}, {high}]")

else:
    print("No unlabeled rows.")

Filled 566 unlabeled train rows.
Recovered range: [0.0000, 3.0000]
Expected range: [0, 3]


In [16]:


# 1. Rebuild feature list
numeric_cols = list(train_df.select_dtypes(include=[np.number]).columns)
features_all = [c for c in numeric_cols if c != target]

# 2. Prepare X and y
X_raw = train_df[features_all].values.astype(float)
y_final = train_df[target].values.astype(float)

# Normalize y (FIXED: was missing before — caused loss to explode to 26000+)
y_mean_final = y_final.mean()
y_std_final  = y_final.std()
if y_std_final == 0:
    y_std_final = 1.0
y_scaled = (y_final - y_mean_final) / y_std_final

# 3. Feature Scaling
X_mean_final = np.mean(X_raw, axis=0)
X_std_final  = np.std(X_raw, axis=0)
X_std_final[X_std_final == 0] = 1.0
X_final = (X_raw - X_mean_final) / X_std_final

# 4. Initialize or remap weights to match features_all length
if 'weights' not in locals() or len(weights) != len(features_all):
    new_w = np.random.randn(len(features_all)) * 0.01
    if 'weights' in locals() and len(weights) < len(features_all):
        new_w[:len(weights)] = weights   # carry over whatever exists
    weights = new_w

bias = 0.0   # reset bias cleanly for retrain

# 5. Retrain
retrain_epochs = CONFIG["epochs"]
stable_learning_rate = CONFIG["learning_rate"]
batch_size = min(CONFIG["batch_size"], len(X_final))

for epoch in range(retrain_epochs):
    perm = np.random.permutation(len(X_final))
    X_sh = X_final[perm]
    y_sh = y_scaled[perm]   # FIXED: use y_scaled not y_final

    for i in range(0, len(X_sh), batch_size):
        X_batch = X_sh[i:i+batch_size]
        y_batch = y_sh[i:i+batch_size]

        y_pred = predict(X_batch, weights, bias)
        error  = y_pred - y_batch

        dw = (2.0 / len(y_batch)) * np.dot(X_batch.T, error)
        db = (2.0 / len(y_batch)) * np.sum(error)

        # Gradient clipping
        dw = np.clip(dw, -1.0, 1.0)
        db = np.clip(db, -1.0, 1.0)

        weights -= stable_learning_rate * dw
        bias    -= stable_learning_rate * db

    if epoch % 50 == 0:
        # FIXED: evaluate loss on y_scaled (same space as predictions)
        current_loss = mse_loss(y_scaled, predict(X_final, weights, bias))
        print(f"Retrain Epoch {epoch:04d} | Loss: {current_loss:.6f}")

# Save normalization stats for use in Cell 86
# These are the ONLY correct stats for de-normalizing predictions after this retrain
print(f"\nRetrain complete.")
print(f"y_mean_final = {y_mean_final:.4f}")
print(f"y_std_final  = {y_std_final:.4f}")
print(f"Weights: {weights}")
print(f"Bias:    {bias:.6f}")

Retrain Epoch 0000 | Loss: 0.680954
Retrain Epoch 0050 | Loss: 0.321700
Retrain Epoch 0100 | Loss: 0.309841
Retrain Epoch 0150 | Loss: 0.307211
Retrain Epoch 0200 | Loss: 0.306503
Retrain Epoch 0250 | Loss: 0.306310

Retrain complete.
y_mean_final = 75.6987
y_std_final  = 12.4683
Weights: [-0.31912653 -0.37326258 -0.30245771 -0.38432177  0.21117405 -0.87620271]
Bias:    -0.000017


In [17]:
#select test rows where hidden is unknown (we evaluate only those)
test_unknown = test_df[test_df[hidden_var].isna()].copy()
print("Test rows (hidden unknown):", len(test_unknown))

Test rows (hidden unknown): 200


In [18]:
# select only known features (exclude hidden + target)

sim_features = [f for f in features_all if f != hidden_var]

print("Similarity features:", sim_features)

Similarity features: ['m1', 'm2', 'm3', 'm4', 'laser']


In [19]:
#normalise_data
labeled_train_sim = train_df[train_df[hidden_var].notna()]
X_train_sim = labeled_train_sim[sim_features].values

sim_mean = X_train_sim.mean(axis=0)
sim_std = X_train_sim.std(axis=0)

sim_std[sim_std == 0] = 1.0  # avoid division error

X_norm = (X_train_sim - sim_mean) / sim_std

In [20]:
# map weights to similarity features

feature_index = {f: i for i, f in enumerate(features_all)}

weights_sim = np.array([weights[feature_index[f]] for f in sim_features])

# normalize weights (importance)

w = np.abs(weights_sim)

if w.sum() == 0:
    w = np.ones_like(w) / len(w)
else:
    w = w / w.sum()

print("Normalized weights:", w)

Normalized weights: [0.20066526 0.23470576 0.19018399 0.24165973 0.13278525]


In [21]:
#SIMILARITY LOSS (WEIGHTED)
def similarity_loss(train_row, test_row, w):
    """ Compute weighted similarity (distance) between two vectors. """
    diff = np.abs(train_row - test_row)
    score = np.sum(w * diff)
    return score

In [22]:
def compute_similarity_scores(test_x):
    """
    Compute similarity scores between one test sample and all training samples (vectorized)."""

    # normalize test point
    test_norm = (test_x - sim_mean) / sim_std

    # vectorized difference (broadcasting)
    diff = np.abs(X_norm - test_norm)   # shape: (n_train, n_features)

    # weighted sum across features
    scores = np.dot(diff, w)

    return scores

In [23]:
def get_hidden_range(nearest_rows):
    """
    Get min and max of hidden variable from nearest rows.
    """
    values = nearest_rows[hidden_var].values
    return np.min(values), np.max(values)


def get_nearest_rows(scores, k=None):
    """
    Select k nearest rows based on similarity scores.
    """
    if k is None:
        k = CONFIG["k_neighbors"]   

    labeled_train = train_df[train_df[hidden_var].notna()].reset_index(drop=True)
    
    nearest_idx = np.argsort(scores)[:k]
    
    return labeled_train.iloc[nearest_idx]

In [24]:
results = []

# use SAME feature space as model
feature_index = {f: i for i, f in enumerate(features)}

if hidden_var not in feature_index:
    raise ValueError(f"{hidden_var} not found in features. Fix feature selection.")

hidden_idx = feature_index[hidden_var]

buffer_pct  = CONFIG["buffer_pct"]
k_neighbors = CONFIG["k_neighbors"]


def eval_at_m5(test_x, m5_val):
    """
    Evaluate model output at a given hidden variable value.
    """
    x = np.zeros(len(features))

    std_safe = np.where(X_std_final == 0, 1.0, X_std_final)

    # fill known features (from similarity space)
    for i, f in enumerate(sim_features):
        fi = feature_index[f]
        x[fi] = (test_x[i] - X_mean_final[fi]) / std_safe[fi]

    # fill hidden variable
    x[hidden_idx] = (m5_val - X_mean_final[hidden_idx]) / std_safe[hidden_idx]

    pred = predict(x.reshape(1, -1), weights, bias)[0]

    return pred * y_std_final + y_mean_final


# MAIN LOOP
for idx, row in test_unknown.iterrows():

    test_x = row[sim_features].values.astype(float)

    # ---- Step 1: find m5 range via similarity ----
    scores = compute_similarity_scores(test_x)
    nearest_rows = get_nearest_rows(scores, k=k_neighbors)

    # 🔥 robust version (better than min/max)
    h_vals = nearest_rows[hidden_var].values
    h_low  = np.percentile(h_vals, 10)
    h_high = np.percentile(h_vals, 90)

    # ---- Step 2: propagate to target ----
    t1 = eval_at_m5(test_x, h_low)
    t2 = eval_at_m5(test_x, h_high)

    base_low  = min(t1, t2)
    base_high = max(t1, t2)

    # symmetric buffer
    width = base_high - base_low
    buffer = buffer_pct * width

    target_low  = base_low  * (1 - buffer_pct)
    target_high = base_high * (1 + buffer_pct)

    # store result
    result = row.to_dict()
    result["hidden_low"]  = round(float(h_low),  4)
    result["hidden_high"] = round(float(h_high), 4)
    result["target_low"]  = round(float(target_low),  4)
    result["target_high"] = round(float(target_high), 4)

    results.append(result)


print(f"Predictions complete: {len(results)} rows")

if len(results) > 0:
    print(
        f"Sample: hidden=[{results[0]['hidden_low']},{results[0]['hidden_high']}]  "
        f"target=[{results[0]['target_low']:.2f},{results[0]['target_high']:.2f}]"
    )

Predictions complete: 200 rows
Sample: hidden=[0.0,3.0]  target=[56.79,96.73]


In [25]:
results_df = pd.DataFrame(results)

results_df.to_csv(CONFIG["output_path"], index=False)

print(f"CSV saved at: {CONFIG['output_path']}")

CSV saved at: D:\project MODEL.T\dataset\model_T_results6.csv


In [26]:
# check condition
results_df["in_range"] = (
    (results_df["target"] >= results_df["target_low"]) &
    (results_df["target"] <= results_df["target_high"])
)

# total rows
total = len(results_df)

# rows inside range
inside = results_df["in_range"].sum()

# percentage
percentage = (inside / total) * 100

print("Total rows:", total)
print("Rows inside range:", inside)
print("Percentage:", percentage, "%")

Total rows: 200
Rows inside range: 169
Percentage: 84.5 %


In [27]:


import numpy as np
import pandas as pd
from sklearn.metrics import r2_score


def predict(X, weights, bias):
    return np.dot(X, weights) + bias

def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def train_linear(X_raw, y_raw, epochs=300, lr=0.001, batch_size=32, seed=42):
    """
    Trains a normalized linear model using SGD.
    Returns: weights, bias, X_mean, X_std, y_mean, y_std
    """
    np.random.seed(seed)

    # Normalize X
    X_mean = X_raw.mean(axis=0)
    X_std  = X_raw.std(axis=0)
    X_std[X_std == 0] = 1.0
    X = (X_raw - X_mean) / X_std

    # Normalize y
    y_mean = y_raw.mean()
    y_std  = y_raw.std()
    if y_std == 0:
        y_std = 1.0
    y = (y_raw - y_mean) / y_std

    n_features = X.shape[1]
    weights = np.random.randn(n_features) * 0.01
    bias    = 0.0
    batch_size = min(batch_size, len(X))

    for epoch in range(epochs):
        perm = np.random.permutation(len(X))
        X_sh = X[perm]
        y_sh = y[perm]
        for i in range(0, len(X_sh), batch_size):
            Xb = X_sh[i:i+batch_size]
            yb = y_sh[i:i+batch_size]
            err = predict(Xb, weights, bias) - yb
            dw = (2.0 / len(yb)) * np.dot(Xb.T, err)
            db = (2.0 / len(yb)) * np.sum(err)
            dw = np.clip(dw, -1.0, 1.0)
            db = np.clip(db, -1.0, 1.0)
            weights -= lr * dw
            bias    -= lr * db

    return weights, bias, X_mean, X_std, y_mean, y_std


def eval_on_test(test_df, features, target, weights, bias,
                 X_mean, X_std, y_mean, y_std, h_fill_value):
    """
    Evaluates R² on test set.
    h_fill_value: the value to use for hidden_var at test time (scalar or array).
    """
    X_test = test_df[features].copy()
    X_test[hidden_var] = h_fill_value   # fill hidden at test time
    X_raw  = X_test.values.astype(float)
    X_std_safe = np.where(X_std == 0, 1.0, X_std)
    X_norm = (X_raw - X_mean) / X_std_safe
    y_pred_norm = predict(X_norm, weights, bias)
    y_pred = y_pred_norm * y_std + y_mean
    y_true = test_df[target].values.astype(float)
    return r2_score(y_true, y_pred), y_pred


# ── Baseline 0: No hidden variable (replicates README baseline) ──
features_no_h = [f for f in features if f != hidden_var]
X_raw_noh = train_df[features_no_h].values.astype(float)
y_raw_noh  = train_df[target].values.astype(float)
w0, b0, Xm0, Xs0, ym0, ys0 = train_linear(
    X_raw_noh, y_raw_noh,
    epochs=CONFIG['epochs'], lr=CONFIG['learning_rate'],
    batch_size=CONFIG['batch_size'], seed=CONFIG['random_seed']
)
X_test_noh = test_df[features_no_h].values.astype(float)
Xs0_safe   = np.where(Xs0 == 0, 1.0, Xs0)
y_pred_noh = (predict((X_test_noh - Xm0) / Xs0_safe, w0, b0)) * ys0 + ym0
r2_no_h    = r2_score(test_df[target].values, y_pred_noh)
print(f"Baseline (no hidden var) R²: {r2_no_h:.4f}")
print()
print("All shared setup complete. Run each baseline cell below independently.")

Baseline (no hidden var) R²: 0.0096

All shared setup complete. Run each baseline cell below independently.


In [28]:
# ── BASELINE 1: MEAN IMPUTATION ───────────────────────────────────────────────

print("=" * 55)
print("  BASELINE 1 — Mean Imputation")
print("=" * 55)

# Step 1 — compute mean from labeled rows only
labeled_mask = train_df[hidden_var].notna()
h_mean = train_df.loc[labeled_mask, hidden_var].mean()
print(f"  Mean of known m5 values : {h_mean:.4f}")
print(f"  (from {labeled_mask.sum()} rule-labeled rows only)")

# Step 2 — fill ALL unlabeled train rows with mean
train_mean = train_df.copy()
train_mean[hidden_var] = train_mean[hidden_var].fillna(h_mean)
assert train_mean[hidden_var].isna().sum() == 0, "Still has NaN after fill"

# Step 3 — train on full training set with imputed h
X_raw_mean = train_mean[features].values.astype(float)
y_raw_mean = train_mean[target].values.astype(float)

w1, b1, Xm1, Xs1, ym1, ys1 = train_linear(
    X_raw_mean, y_raw_mean,
    epochs=CONFIG['epochs'],
    lr=CONFIG['learning_rate'],
    batch_size=CONFIG['batch_size'],
    seed=CONFIG['random_seed']
)

# Step 4 — predict on test set (fill test m5 with same mean)
r2_mean, y_pred_mean = eval_on_test(
    test_df, features, target,
    w1, b1, Xm1, Xs1, ym1, ys1,
    h_fill_value=h_mean
)

print(f"\n  Test R² (Mean Imputation) : {r2_mean:.4f}")
print(f"  Compare → No-h baseline  : {r2_no_h:.4f}")
print(f"  Compare → Model-T        : [your Model-T R² here]")
print("=" * 55)

  BASELINE 1 — Mean Imputation
  Mean of known m5 values : 1.4876
  (from 800 rule-labeled rows only)

  Test R² (Mean Imputation) : -0.0921
  Compare → No-h baseline  : 0.0096
  Compare → Model-T        : [your Model-T R² here]


In [29]:
# ── BASELINE 2: KNN IMPUTATION ────────────────────────────────────────────────

print("=" * 55)
print("  BASELINE 2 — KNN Imputation")
print("=" * 55)

k = CONFIG['k_neighbors']   # 10 neighbors — same as Model-T similarity search

# Observed features only (no hidden_var, no target)
obs_features = [f for f in features if f != hidden_var]

# Labeled pool — these have known m5 values
# FIX: Use clean snapshot (train_df has h already recovered by Model-T)
labeled_pool   = train_df_clean[train_df_clean[hidden_var].notna()].copy()
unlabeled_pool = train_df_clean[train_df_clean[hidden_var].isna()].copy()

X_labeled_obs = labeled_pool[obs_features].values.astype(float)
h_labeled     = labeled_pool[hidden_var].values.astype(float)

# Normalize using labeled pool statistics
obs_mean = X_labeled_obs.mean(axis=0)
obs_std  = X_labeled_obs.std(axis=0)
obs_std[obs_std == 0] = 1.0
X_labeled_norm = (X_labeled_obs - obs_mean) / obs_std

def knn_impute_h(X_query_raw, X_labeled_norm, h_labeled, obs_mean, obs_std, k):
    """
    For each row in X_query_raw, find k nearest labeled neighbors
    and return mean of their h values.
    """
    X_query_norm = (X_query_raw - obs_mean) / obs_std
    h_imputed = np.zeros(len(X_query_norm))

    for i, q in enumerate(X_query_norm):
        # Euclidean distance to all labeled rows
        dists = np.sqrt(np.sum((X_labeled_norm - q) ** 2, axis=1))
        nearest_idx = np.argsort(dists)[:k]
        h_imputed[i] = h_labeled[nearest_idx].mean()

    return h_imputed

# Impute unlabeled train rows
X_unlabeled_obs = unlabeled_pool[obs_features].values.astype(float)
print(f"  Imputing {len(X_unlabeled_obs)} unlabeled train rows (k={k})...")
h_knn_unlabeled = knn_impute_h(
    X_unlabeled_obs, X_labeled_norm, h_labeled, obs_mean, obs_std, k
)

# Clip to valid range from rules
h_low_clip, h_high_clip = rules['variables'][hidden_var]['range']
h_knn_unlabeled = np.clip(h_knn_unlabeled, h_low_clip, h_high_clip)

# Build full training set with imputed h
train_knn = train_df_clean.copy()  # FIX: start from clean copy
train_knn.loc[unlabeled_pool.index, hidden_var] = h_knn_unlabeled
assert train_knn[hidden_var].isna().sum() == 0
print(f"  Recovered h range: [{h_knn_unlabeled.min():.3f}, {h_knn_unlabeled.max():.3f}]")

# Train
X_raw_knn = train_knn[features].values.astype(float)
y_raw_knn  = train_knn[target].values.astype(float)
w2, b2, Xm2, Xs2, ym2, ys2 = train_linear(
    X_raw_knn, y_raw_knn,
    epochs=CONFIG['epochs'],
    lr=CONFIG['learning_rate'],
    batch_size=CONFIG['batch_size'],
    seed=CONFIG['random_seed']
)

# Impute test rows too — use KNN from labeled pool
X_test_obs = test_df[obs_features].values.astype(float)
print(f"  Imputing {len(X_test_obs)} test rows...")
h_knn_test = knn_impute_h(
    X_test_obs, X_labeled_norm, h_labeled, obs_mean, obs_std, k
)
h_knn_test = np.clip(h_knn_test, h_low_clip, h_high_clip)

r2_knn, y_pred_knn = eval_on_test(
    test_df, features, target,
    w2, b2, Xm2, Xs2, ym2, ys2,
    h_fill_value=h_knn_test
)

print(f"\n  Test R² (KNN Imputation) : {r2_knn:.4f}")
print(f"  Compare → No-h baseline : {r2_no_h:.4f}")
print(f"  Compare → Mean baseline : {r2_mean:.4f}")
print(f"  Compare → Model-T       : [your Model-T R² here]")
print("=" * 55)

  BASELINE 2 — KNN Imputation
  Imputing 566 unlabeled train rows (k=10)...
  Recovered h range: [0.900, 1.900]
  Imputing 200 test rows...

  Test R² (KNN Imputation) : 0.0155
  Compare → No-h baseline : 0.0096
  Compare → Mean baseline : -0.0921
  Compare → Model-T       : [your Model-T R² here]


In [30]:
# ── BASELINE 3: MICE ──────────────────────────────────────────────────────────

from sklearn.experimental import enable_iterative_imputer   # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

print("=" * 55)
print("  BASELINE 3 — MICE Imputation")
print("=" * 55)

# MICE imputes NaN values using all other columns as predictors.
# We fit on train_df (features + hidden_var), then transform to get filled h.
# target column is excluded — MICE should not see the output.

mice_cols = features   # includes hidden_var (m5) as the column to impute

# Fit MICE on training data
mice_imputer = IterativeImputer(
    estimator=BayesianRidge(),   # default MICE estimator
    max_iter=10,
    random_state=CONFIG['random_seed'],
    verbose=0
)

# Train matrix: only feature columns (no target)
# FIX: Use clean snapshot (not train_df which has recovered h)
train_mice_matrix = train_df_clean[mice_cols].copy().values.astype(float)

print("  Fitting MICE on training data...")
mice_imputer.fit(train_mice_matrix)

# Transform training data
train_mice_filled = mice_imputer.transform(train_mice_matrix)
train_mice_df = pd.DataFrame(train_mice_filled, columns=mice_cols)

# Clip imputed h to valid range
h_low_clip, h_high_clip = rules['variables'][hidden_var]['range']
train_mice_df[hidden_var] = train_mice_df[hidden_var].clip(h_low_clip, h_high_clip)

h_mice_recovered = train_mice_df[hidden_var].values[
    train_df_clean[hidden_var].isna().values  # FIX: use clean mask
]
print(f"  Recovered h range (unlabeled): [{h_mice_recovered.min():.3f}, {h_mice_recovered.max():.3f}]")

# Train model
X_raw_mice = train_mice_df[features].values.astype(float)
y_raw_mice  = train_df_clean[target].values.astype(float)   # target unchanged

w3, b3, Xm3, Xs3, ym3, ys3 = train_linear(
    X_raw_mice, y_raw_mice,
    epochs=CONFIG['epochs'],
    lr=CONFIG['learning_rate'],
    batch_size=CONFIG['batch_size'],
    seed=CONFIG['random_seed']
)

# Impute test rows — stack with train so MICE has context
# Test rows have m5 = NaN always
test_mice_matrix = test_df[mice_cols].copy().values.astype(float)

# MICE transform on test rows directly (fitted on train)
print("  Imputing test rows with MICE...")
test_mice_filled = mice_imputer.transform(test_mice_matrix)
test_mice_df = pd.DataFrame(test_mice_filled, columns=mice_cols)
test_mice_df[hidden_var] = test_mice_df[hidden_var].clip(h_low_clip, h_high_clip)
h_mice_test = test_mice_df[hidden_var].values

r2_mice, y_pred_mice = eval_on_test(
    test_df, features, target,
    w3, b3, Xm3, Xs3, ym3, ys3,
    h_fill_value=h_mice_test
)

print(f"\n  Test R² (MICE)           : {r2_mice:.4f}")
print(f"  Compare → No-h baseline : {r2_no_h:.4f}")
print(f"  Compare → Mean baseline : {r2_mean:.4f}")
print(f"  Compare → KNN baseline  : {r2_knn:.4f}")
print(f"  Compare → Model-T       : [your Model-T R² here]")
print("=" * 55)

  BASELINE 3 — MICE Imputation
  Fitting MICE on training data...
  Recovered h range (unlabeled): [1.280, 1.758]
  Imputing test rows with MICE...

  Test R² (MICE)           : 0.0118
  Compare → No-h baseline : 0.0096
  Compare → Mean baseline : -0.0921
  Compare → KNN baseline  : 0.0155
  Compare → Model-T       : [your Model-T R² here]


In [33]:
# ── BASELINE 4: EM ALGORITHM ──────────────────────────────────────────────────

from scipy.stats import mode as scipy_mode
from sklearn.neighbors import NearestNeighbors

print("=" * 55)
print("  BASELINE 4 — EM Algorithm")
print("=" * 55)

# Safety: rebuild clean snapshot if needed
if 'train_df_clean' not in dir() or train_df_clean[hidden_var].isna().sum() == 0:
    _raw = train_df.copy()
    _raw[hidden_var] = np.nan
    for rule in rules.get('rules', []):
        _mask = _raw.eval(rule['condition'])
        for _var, _val in rule['assign'].items():
            _raw.loc[_mask & _raw[_var].isna(), _var] = _val
    train_df_clean = _raw
    print('  [Info] Rebuilt train_df_clean from rules')

# Discrete h values from RULE-LABELED rows only (not recovered values)
h_low_clip, h_high_clip = rules['variables'][hidden_var]['range']
h_values = np.array([float(v) for v in
    sorted(train_df_clean[train_df_clean[hidden_var].notna()][hidden_var].unique())
])
print(f"  Discrete h values found : {h_values}")
print(f"  (should be 4 clean values: 0, 1, 2, 3)")

labeled_mask_em   = train_df_clean[hidden_var].notna()
unlabeled_mask_em = train_df_clean[hidden_var].isna()
n_unlabeled       = unlabeled_mask_em.sum()
print(f"  Unlabeled rows to fill  : {n_unlabeled}")

# ── Initialize: fill unlabeled with mean h (warm start) ──
train_em = train_df_clean.copy()
h_init   = train_df_clean.loc[labeled_mask_em, hidden_var].mean()
train_em.loc[unlabeled_mask_em, hidden_var] = h_init

max_em_iter = 20
em_lr        = CONFIG['learning_rate']
em_epochs    = 150
em_batch     = CONFIG['batch_size']

prev_h_assignments = None

for em_iter in range(max_em_iter):

    # ── M-STEP: train model on all rows with current h ──
    X_em_raw = train_em[features].values.astype(float)
    y_em_raw = train_em[target].values.astype(float)

    w_em, b_em, Xm_em, Xs_em, ym_em, ys_em = train_linear(
        X_em_raw, y_em_raw,
        epochs=em_epochs, lr=em_lr,
        batch_size=em_batch, seed=CONFIG['random_seed']
    )

    # ── E-STEP (TRAINING): pick h that minimises residual for unlabeled rows ──
    # Valid at train time — we are using training y values, not test y values
    unlabeled_rows_em = train_em[unlabeled_mask_em].copy()
    Xs_em_safe = np.where(Xs_em == 0, 1.0, Xs_em)

    new_h = np.zeros(n_unlabeled)

    for i, (row_idx, row) in enumerate(unlabeled_rows_em.iterrows()):
        best_h    = h_values[0]
        best_loss = np.inf
        y_true_i  = row[target]   # OK at TRAIN time — this is training data

        for h_val in h_values:
            x_row = row[features].copy()
            x_row[hidden_var] = h_val
            x_raw = x_row.values.astype(float).reshape(1, -1)
            x_norm = (x_raw - Xm_em) / Xs_em_safe
            y_hat  = predict(x_norm, w_em, b_em)[0] * ys_em + ym_em
            loss   = (y_true_i - y_hat) ** 2

            if loss < best_loss:
                best_loss = loss
                best_h    = h_val

        new_h[i] = best_h

    train_em.loc[unlabeled_mask_em, hidden_var] = new_h

    # ── Convergence check ──
    if prev_h_assignments is not None:
        n_changed = np.sum(new_h != prev_h_assignments)
        print(f"  EM iter {em_iter+1:02d} — h assignments changed: {n_changed} / {n_unlabeled}")
        if n_changed == 0:
            print(f"  Converged at iteration {em_iter+1}.")
            break
    else:
        print(f"  EM iter {em_iter+1:02d} — initial M-step complete")

    prev_h_assignments = new_h.copy()

else:
    print(f"  EM reached max iterations ({max_em_iter}) without full convergence.")


# ── Test-time prediction — NO LEAKAGE ──────────────────────────────────────
# y_true is unknown at test time. Find k nearest training neighbors
# by observed features only, take mode of their converged h values.

print("\n  Predicting test set (KNN mode on converged h — no leakage)...")

obs_feats = [f for f in features if f != hidden_var]
obs_idx   = [i for i, f in enumerate(features) if f != hidden_var]
Xm_obs    = Xm_em[obs_idx]
Xs_obs    = np.where(Xs_em[obs_idx] == 0, 1.0, Xs_em[obs_idx])

X_train_obs  = train_em[obs_feats].values.astype(float)
X_train_norm = (X_train_obs - Xm_obs) / Xs_obs
h_train_conv = train_em[hidden_var].values

nn_em = NearestNeighbors(n_neighbors=CONFIG['k_neighbors'])
nn_em.fit(X_train_norm)

X_test_obs       = test_df[obs_feats].values.astype(float)
X_test_norm      = (X_test_obs - Xm_obs) / Xs_obs
_, neighbor_idxs = nn_em.kneighbors(X_test_norm)

h_em_test = np.array([
    scipy_mode(h_train_conv[neighbor_idxs[i]], keepdims=False).mode
    for i in range(len(test_df))
])
h_em_test = np.clip(h_em_test, h_low_clip, h_high_clip)

print(f"  h assigned at test time — unique values: {np.unique(h_em_test)}")

r2_em, y_pred_em = eval_on_test(
    test_df, features, target,
    w_em, b_em, Xm_em, Xs_em, ym_em, ys_em,
    h_fill_value=h_em_test
)

print(f"\n  Test R² (EM — no leakage) : {r2_em:.4f}")
print(f"  Compare → No-h baseline   : {r2_no_h:.4f}")
print(f"  Compare → Mean baseline   : {r2_mean:.4f}")
print(f"  Compare → KNN baseline    : {r2_knn:.4f}")
print(f"  Compare → MICE            : {r2_mice:.4f}")
print(f"  Compare → Model-T         : [your Model-T R² here]")
print("=" * 55)

  BASELINE 4 — EM Algorithm
  Discrete h values found : [0. 1. 2. 3.]
  (should be 4 clean values: 0, 1, 2, 3)
  Unlabeled rows to fill  : 566
  EM iter 01 — initial M-step complete
  EM iter 02 — h assignments changed: 171 / 566
  EM iter 03 — h assignments changed: 66 / 566
  EM iter 04 — h assignments changed: 26 / 566
  EM iter 05 — h assignments changed: 12 / 566
  EM iter 06 — h assignments changed: 2 / 566
  EM iter 07 — h assignments changed: 1 / 566
  EM iter 08 — h assignments changed: 0 / 566
  Converged at iteration 8.

  Predicting test set (KNN mode on converged h — no leakage)...
  h assigned at test time — unique values: [0. 1. 2. 3.]

  Test R² (EM — no leakage) : -0.6286
  Compare → No-h baseline   : 0.0096
  Compare → Mean baseline   : -0.0921
  Compare → KNN baseline    : 0.0155
  Compare → MICE            : 0.0118
  Compare → Model-T         : [your Model-T R² here]


In [32]:
# ── FINAL COMPARISON TABLE ────────────────────────────────────────────────────

# Replace the Model-T R² below with your actual value from the pipeline.
# Model-T doesn't produce a single R² — it produces intervals.
# To get R²: use midpoint of [target_low, target_high] as point estimate.

results_df_loaded = pd.read_csv(CONFIG['output_path'])
y_true_test  = results_df_loaded['target'].values
y_mid_model_t = (results_df_loaded['target_low'].values +
                 results_df_loaded['target_high'].values) / 2
r2_model_t   = r2_score(y_true_test, y_mid_model_t)

print("\n" + "=" * 55)
print("  FINAL COMPARISON TABLE")
print("=" * 55)
print(f"  {'Method':<28} {'R²':>8}")
print("  " + "-" * 38)
print(f"  {'No hidden var (baseline)':<28} {r2_no_h:>8.4f}")
print(f"  {'Mean imputation':<28} {r2_mean:>8.4f}")
print(f"  {'KNN imputation (k=10)':<28} {r2_knn:>8.4f}")
print(f"  {'MICE':<28} {r2_mice:>8.4f}")
print(f"  {'EM algorithm':<28} {r2_em:>8.4f}")
print(f"  {'Model-T (interval midpoint)':<28} {r2_model_t:>8.4f}")
print("  " + "-" * 38)
print(f"  {'Oracle (true h known)':<28} {'~0.50':>8}  [README target]")
print("=" * 55)
print()
print("  Coverage (Model-T only):")
in_range = (
    (results_df_loaded['target'] >= results_df_loaded['target_low']) &
    (results_df_loaded['target'] <= results_df_loaded['target_high'])
)
print(f"  {in_range.sum()} / {len(in_range)} rows inside interval  "
      f"({in_range.mean()*100:.1f}%)")


  FINAL COMPARISON TABLE
  Method                             R²
  --------------------------------------
  No hidden var (baseline)       0.0096
  Mean imputation               -0.0921
  KNN imputation (k=10)          0.0155
  MICE                           0.0118
  EM algorithm                   0.9712
  Model-T (interval midpoint)    0.0251
  --------------------------------------
  Oracle (true h known)           ~0.50  [README target]

  Coverage (Model-T only):
  169 / 200 rows inside interval  (84.5%)
